# 3. Interaction benchmark notebook

这个 notebook 检查 `interaction.py`，重点不是 FRG flow，而是 bare vertex 本身有没有明显的物理错误。

这里用一个最简单的 **spinful but trivial** 的模型：

- `KagomeKaneMeleSOC(t=1, l1=0, l2=0, spin=True, B=0)`
- 然后对 up / down 两个 3x3 block 分别 patch

这样做的好处是：

- 两个 spin sector 在物理上应完全等价
- bare extended Hubbard 的 spin 选择规则应该最清楚


In [1]:
import os, sys, math, cmath
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, '/mnt/data')

from noninteracting import KagomeNagaosa, KagomeStaggerFlux, KagomeKaneMeleSOC
from patching import FSPatcher
from interaction import BareExtendedHubbard
from channels import ChannelDecomposer

RESULTS = []

def record(name, passed, detail='', value=None):
    RESULTS.append({
        'test': name,
        'passed': bool(passed),
        'detail': str(detail),
        'value': value,
    })

def check(name, cond, detail_ok='', detail_fail='', value=None):
    if cond:
        record(name, True, detail_ok, value)
    else:
        record(name, False, detail_fail, value)


def summarize_results():
    n_pass = sum(r['passed'] for r in RESULTS)
    n_fail = len(RESULTS) - n_pass
    print('='*90)
    print(f'Total tests: {len(RESULTS)} | PASS: {n_pass} | FAIL: {n_fail}')
    print('='*90)
    for i, r in enumerate(RESULTS, 1):
        flag = 'PASS' if r['passed'] else 'FAIL'
        print(f"[{i:02d}] {flag:4s} | {r['test']}")
        if r['detail']:
            print('     ', r['detail'])
        if r['value'] is not None:
            print('     value =', r['value'])
    print('='*90)
    return n_fail


In [2]:
# 你可以改这里
model = KagomeKaneMeleSOC(parameters={'t': 1.0, 'l1': 0.0, 'l2': 0.0}, spin=True, B=0.0)
mu = model.EF_from_filling(5/12, N=100, tol=1e-4)
print('mu =', mu)

patcher_up = FSPatcher(model=model, band_index=1, mu=mu, Npatch=16, grid_size=220, orbital_slice=slice(0,3), verbose=False)
patcher_dn = FSPatcher(model=model, band_index=1, mu=mu, Npatch=16, grid_size=220, orbital_slice=slice(3,6), verbose=False)

ps_up = patcher_up.build()
ps_dn = patcher_dn.build()
patchsets = {'up': ps_up, 'dn': ps_dn}

interaction = BareExtendedHubbard.from_kagome_model(model, U=2.0, V=1.0)
print('interaction =', interaction)


mu = -3.0083462602747215e-17
interaction = BareExtendedHubbard(U=2.0, V=1.0, delta1=array([0.5      , 0.8660254]), delta2=array([ 0.5      , -0.8660254]), delta3=array([-1.,  0.]))


In [3]:
# 1) up/down patchsets should match for this trivial spinful model
ks_up = np.array([p.k_cart for p in ps_up.patches])
ks_dn = np.array([p.k_cart for p in ps_dn.patches])
d = np.linalg.norm(ks_up - ks_dn, axis=1)
check(
    'trivial model: up/down patchsets coincide',
    d.max() < 1e-8,
    detail_ok=f'max |k_up-k_dn| = {d.max():.3e}',
    detail_fail=f'up/down patchsets differ unexpectedly, max diff = {d.max():.3e}',
    value={'max': float(d.max()), 'mean': float(d.mean())},
)


In [4]:
# pick a few patch labels
p1, p2, p3, p4 = 0, 1, 2, 3


In [5]:
# 2) direct density-density vertex should conserve spin along each fermion line
# allowed: s3=s1 and s4=s2
# forbidden: line-flipping combinations should give 0 in the DIRECT vertex
cases = [
    ('allowed up,dn->up,dn', ('up','dn','up','dn'), True),
    ('forbidden up,dn->dn,up', ('up','dn','dn','up'), False),
    ('forbidden up,up->dn,up', ('up','up','dn','up'), False),
    ('forbidden dn,up->dn,dn', ('dn','up','dn','dn'), False),
]

for name, spins, should_be_allowed in cases:
    s1,s2,s3,s4 = spins
    val = interaction.patch_vertex(patchsets, p1,s1,p2,s2,p3,s3,p4,s4, antisym=False, check_momentum=False)
    mag = abs(val)
    if should_be_allowed:
        check(name, mag > 1e-10, detail_ok=f'|V_dir|={mag:.3e}', detail_fail=f'allowed process came out ~0: |V_dir|={mag:.3e}', value=float(mag))
    else:
        check(name, mag < 1e-12, detail_ok=f'forbidden process correctly vanished: |V_dir|={mag:.3e}', detail_fail=f'forbidden direct process is nonzero: |V_dir|={mag:.3e}', value=float(mag))


In [6]:
# 3) onsite U contributes only for opposite-spin DIRECT scattering when V=0
interaction_U = BareExtendedHubbard.from_kagome_model(model, U=2.0, V=0.0)
val_ud = interaction_U.patch_vertex(patchsets, p1,'up',p2,'dn',p3,'up',p4,'dn', antisym=False, check_momentum=False)
val_uu = interaction_U.patch_vertex(patchsets, p1,'up',p2,'up',p3,'up',p4,'up', antisym=False, check_momentum=False)
check('U-only opposite-spin direct nonzero', abs(val_ud) > 1e-10, detail_ok=f'|V_ud|={abs(val_ud):.3e}', detail_fail=f'U-only opposite-spin direct too small: |V_ud|={abs(val_ud):.3e}', value=float(abs(val_ud)))
check('U-only same-spin direct zero', abs(val_uu) < 1e-12, detail_ok=f'|V_uu|={abs(val_uu):.3e}', detail_fail=f'U-only same-spin direct should vanish but got {abs(val_uu):.3e}', value=float(abs(val_uu)))


In [7]:
# 4) V-only allows both same-spin and opposite-spin DIRECT scattering
interaction_V = BareExtendedHubbard.from_kagome_model(model, U=0.0, V=1.0)
val_ud = interaction_V.patch_vertex(patchsets, p1,'up',p2,'dn',p3,'up',p4,'dn', antisym=False, check_momentum=False)
val_uu = interaction_V.patch_vertex(patchsets, p1,'up',p2,'up',p3,'up',p4,'up', antisym=False, check_momentum=False)
check('V-only opposite-spin direct nonzero', abs(val_ud) > 1e-10, detail_ok=f'|V_ud|={abs(val_ud):.3e}', detail_fail=f'V-only opposite-spin direct too small: |V_ud|={abs(val_ud):.3e}', value=float(abs(val_ud)))
check('V-only same-spin direct nonzero', abs(val_uu) > 1e-10, detail_ok=f'|V_uu|={abs(val_uu):.3e}', detail_fail=f'V-only same-spin direct too small: |V_uu|={abs(val_uu):.3e}', value=float(abs(val_uu)))


In [8]:
# 5) antisymmetry under exchanging outgoing legs 3 <-> 4
# Γ(1,2->3,4) = - Γ(1,2->4,3)
G34 = interaction.patch_vertex(patchsets, p1,'up',p2,'dn',p3,'up',p4,'dn', antisym=True, check_momentum=False)
G43 = interaction.patch_vertex(patchsets, p1,'up',p2,'dn',p4,'dn',p3,'up', antisym=True, check_momentum=False)
err = abs(G34 + G43)
check(
    'antisymmetry under 3<->4',
    err < 1e-10,
    detail_ok=f'|Γ(1234)+Γ(1243)|={err:.3e}',
    detail_fail=f'antisymmetry broken: error={err:.3e}',
    value=float(err),
)


In [9]:
# 6) momentum conservation checker should reject generic invalid tuples (usually)
# 先找一组明显不守恒的 patch tuple，然后确认 check_momentum=True 时会报错
found_invalid = False
for a in range(ps_up.Npatch):
    for b in range(ps_dn.Npatch):
        for c in range(ps_up.Npatch):
            for d_ in range(ps_dn.Npatch):
                try:
                    interaction.patch_vertex(patchsets, a,'up',b,'dn',c,'up',d_,'dn', antisym=False, check_momentum=True)
                except ValueError:
                    found_invalid = True
                    bad_tuple = (a,b,c,d_)
                    break
            if found_invalid:
                break
        if found_invalid:
            break
    if found_invalid:
        break

check(
    'momentum checker catches invalid tuple',
    found_invalid,
    detail_ok=f'found invalid tuple {bad_tuple} that raises ValueError',
    detail_fail='did not find any invalid tuple; this is suspicious for generic patch representatives',
    value=bad_tuple if found_invalid else None,
)


In [10]:
# 7) gauge robustness test: multiply each Bloch vector by arbitrary phase; physical projected vertex should stay unchanged
rng = np.random.default_rng(1234)
phases = np.exp(1j * 2*np.pi * rng.random(ps_up.Npatch))

class FakePatch:
    def __init__(self, p, phase):
        self.k_cart = p.k_cart
        self.eigvec = p.eigvec * phase
        self.k_red = p.k_red
        self.energy = p.energy
        self.vF = p.vF
        self.vF_norm = p.vF_norm
        self.orbital_weight = p.orbital_weight
        self.patch_id = p.patch_id

class FakePatchSet:
    def __init__(self, ps, phases):
        self.mu = ps.mu
        self.mu_used_for_contour = ps.mu_used_for_contour
        self.band_index = ps.band_index
        self.filling = ps.filling
        self.patches = [FakePatch(p, phases[i]) for i,p in enumerate(ps.patches)]
        self.fs_contour_k = ps.fs_contour_k
        self.bz_vertices = ps.bz_vertices
        self.b1 = ps.b1
        self.b2 = ps.b2
        self.Npatch = len(self.patches)

ps_up_phase = FakePatchSet(ps_up, phases)
ps_dn_phase = FakePatchSet(ps_dn, phases)
patchsets_phase = {'up': ps_up_phase, 'dn': ps_dn_phase}

val0 = interaction.patch_vertex(patchsets, p1,'up',p2,'dn',p3,'up',p4,'dn', antisym=True, check_momentum=False)
val1 = interaction.patch_vertex(patchsets_phase, p1,'up',p2,'dn',p3,'up',p4,'dn', antisym=True, check_momentum=False)
err = abs(val0 - val1)
check(
    'vertex invariant under patchwise Bloch phase gauge',
    err < 1e-10,
    detail_ok=f'|Γ-Γ_phase|={err:.3e}',
    detail_fail=f'projected vertex changed under gauge phase: err={err:.3e}',
    value=float(err),
)


In [11]:
summarize_results()


Total tests: 12 | PASS: 11 | FAIL: 1
[01] PASS | trivial model: up/down patchsets coincide
      max |k_up-k_dn| = 0.000e+00
     value = {'max': 0.0, 'mean': 0.0}
[02] PASS | allowed up,dn->up,dn
      |V_dir|=4.308e-01
     value = 0.4308189270167951
[03] PASS | forbidden up,dn->dn,up
      forbidden process correctly vanished: |V_dir|=0.000e+00
     value = 0.0
[04] PASS | forbidden up,up->dn,up
      forbidden process correctly vanished: |V_dir|=0.000e+00
     value = 0.0
[05] PASS | forbidden dn,up->dn,dn
      forbidden process correctly vanished: |V_dir|=0.000e+00
     value = 0.0
[06] PASS | U-only opposite-spin direct nonzero
      |V_ud|=1.834e-02
     value = 0.018336298911987084
[07] PASS | U-only same-spin direct zero
      |V_uu|=0.000e+00
     value = 0.0
[08] PASS | V-only opposite-spin direct nonzero
      |V_ud|=4.125e-01
     value = 0.41248262810480796
[09] PASS | V-only same-spin direct nonzero
      |V_uu|=4.125e-01
     value = 0.41248262810480796
[10] PASS | ant

1